# Tensors: The Foundation of PyTorch

Reach for this when you need: 
- Quick syntax for tensor creation, manipulation, and dtypes.
- Reference for broadcasting rules and reshaping.
- Understanding memory layouts (contiguous vs non-contiguous).

In [1]:
import torch
import numpy as np

# Global device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Tensor Creation and Dtypes

### Basic Factories

In [2]:
# torch.empty: Uninitialized memory. Fast but contains garbage data.
x = torch.empty(2, 3)

# torch.zeros/ones: Initialize with 0s or 1s. Industry standard for bias/weight init.
zeros = torch.zeros(5, 5)
ones = torch.ones(5, 5, dtype=torch.float32)

# torch.rand: Uniform distribution [0, 1). Standard for random init.
rand = torch.rand(2, 2)

# torch.tensor: Create from list/existing data. Infers dtype unless specified.
data = torch.tensor([[1, 2], [3, 4]], dtype=torch.float64)

### Dtype Conversion and Casting

| Dtype | Usage | Precision |
| :--- | :--- | :--- |
| `torch.float32` | Standard DL (float) | 32-bit |
| `torch.float16` | Mixed precision (half) | 16-bit |
| `torch.bfloat16` | Brain Float (Ampere+) | 16-bit |
| `torch.int64` | Classification labels (long) | 64-bit |

In [9]:
x = torch.ones(2, 2)

# Using .to() is the preferred way to cast and move to device simultaneously
x_half = x.to(torch.float16)
x_gpu = x.to(device)

# .type() is legacy, avoid in new codebases
x_long = x.long() # Shorthand for x.to(torch.int64)

## 2. Manipulation: Shapes and Views

### Reshaping and Transposing

**torch.view vs torch.reshape**
- `view`: Returns a view of the original data. Fails if memory is not contiguous.
- `reshape`: Same as `view` but clones data if necessary (safer).

✅ **Use when**: Changing feature dimensions before a Linear layer.
❌ **Don't use when**: You need to swap logical axes (use `transpose` or `permute`).

In [17]:
x = torch.randn(4, 4)

# Use -1 to infer the dimension automatically
y = x.view(16)
z = x.view(-1, 8) # result: [2, 8]

# Transpose: Swap two dimensions
t = x.transpose(0, 1)

# Permute: Reorder all dimensions (common for NCHW to NHWC conversion)
p = x.unsqueeze(0).permute(0, 2, 1)

### Squeezing and Unsqueezing

**torch.unsqueeze(dim)** / **torch.squeeze()**
- `unsqueeze`: Adds a singleton dimension (size 1) at `dim`.
- `squeeze`: Removes all dimensions of size 1 (or specific `dim`).

In [ ]:
x = torch.zeros(1, 10)

# Add batch dimension: [1, 10] -> [1, 1, 10]
y = x.unsqueeze(0)

# Remove singleton dimensions: [1, 1, 10] -> [10]
z = y.squeeze()

In [34]:
z

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## 3. Broadcasting Rules

Broadcasting allows operations between tensors of different shapes if:
1. Dimensions match exactly.
2. One of the dimensions is 1 (expanded to match the other).

✅ **Use when**: Adding a bias vector to a batch of activations.
❌ **Don't use when**: You want explicit control (use `expand` or `repeat` for clarity).

In [ ]:
a = torch.ones(3, 1)
b = torch.ones(1, 3)

# Result will be (3, 3) because both are expanded
result = a + b

# Use expand() to see what broadcasting would do without copying data
a_expanded = a.expand(3, 3)

### Key Takeaways
- Always prefer `view` for speed, but use `reshape` if you encounter non-contiguous memory errors.
- Tensor operations are generally O(1) for shape changes (metadata only) until data is copied.
- Use `torch.device` to keep code hardware-agnostic from the start.
- Dtype `torch.float32` is the industry standard default; `bfloat16` is the standard for modern LLM training.